In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/13 16:25:10 WARN Utils: Your hostname, codespaces-dff4ee, resolves to a loopback address: 127.0.0.1; using 10.0.3.32 instead (on interface eth0)
26/03/13 16:25:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 16:25:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/13 16:25:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:

!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz


--2026-03-13 16:25:45--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.26.156.215
Connecting to github.com (github.com)|20.26.156.215|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-13T17%3A25%3A38Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-13T16%3A24%3A40Z&ske=2026-03-13T17%3A25%3A38Z&sks=b&skv=2018-11-09&sig=rO4BbZdOOhSMhPJUa4KBfCi5OGykVkEPQZ0FB8BUO4c%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzQyMjc0NiwibmJmIjoxNzczNDE5MTQ2LCJwYXRoIj

In [ ]:

!gzip -dc fhvhv_tripdata_2021-01.csv.gz


hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
HV0003,B02875,2021-01-01 00:40:12,2021-01-01 00:53:31,255,232,
HV0003,B02875,2021-01-01 00:56:45,2021-01-01 01:17:42,232,198,
HV0003,B02835,2021-01-01 00:29:04,2021-01-01 00:36:27,113,48,
HV0003,B02835,2021-01-01 00:48:56,2021-01-01 00:59:12,239,75,
HV0004,B02800,2021-01-01 00:15:24,2021-01-01 00:38:31,181,237,
HV0004,B02800,2021-

In [6]:
df = spark.read.option("header","true").csv("fhvhv_tripdata_2021-01.csv.gz")


In [7]:

df.schema


StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [8]:
from pyspark.sql import types


In [9]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [10]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv.gz")


In [11]:
df = df.repartition(24)

In [12]:
df.write.parquet('fhvhv/2021/01/')

In [13]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [14]:
from pyspark.sql import functions as F

In [15]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [16]:
crazy_stuff('B02884')

's/b44'

In [17]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())


In [18]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

[Stage 6:>                                                          (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-09|  2021-01-09|          85|          35|
|  e/acc| 2021-01-07|  2021-01-07|          37|          36|
|  e/b38| 2021-01-22|  2021-01-22|         158|         229|
|  e/9ce| 2021-01-20|  2021-01-20|         216|         130|
|  e/9ce| 2021-01-29|  2021-01-29|         160|          48|
|  e/b3b| 2021-01-15|  2021-01-15|         165|          91|
|  e/9ce| 2021-01-02|  2021-01-02|         262|         239|
|  e/b38| 2021-01-20|  2021-01-20|          85|          67|
|  e/acc| 2021-01-17|  2021-01-17|          48|          47|
|  s/acd| 2021-01-17|  2021-01-17|         244|         159|
|  e/acc| 2021-01-21|  2021-01-21|         244|         242|
|  e/9ce| 2021-01-23|  2021-01-23|         234|          12|
|  e/b3c| 2021-01-22|  2021-01-22|         143|         239|
|  s/b13| 2021-01-23|  2

In [19]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]